# 00 — One-time similarity-pipeline reset

Run this **once, immediately before the first batch using the new notebook 05**. Keep other pipeline notebooks stopped during the reset.

The next cell defaults to `RESET_MODE = "preview"`: it lists exact output tables, row counts, and active MLflow runs without changing them. Inspect the inventory, then change that value to `"execute"` and run the cell once. Return it to `"preview"` after success.

The explicit allowlist covers derived outputs from notebooks 05/06/07 and the new similarity status tables. Execute mode deletes active runs in the three named pipeline MLflow experiments, retaining their containers, and drops those output tables. It does not touch AI attributes, Sales, Rental, or Eilers source tables. It does not clear unrelated workspace experiments or issue a physical-storage purge.

Table drops start fresh schemas and Delta transaction histories. The new processing notebooks recreate their outputs as needed. Clearing results without clearing status records can leave stale completion state; resetting one stage alone is intentionally not the default.

If any inventory or reset step fails, resolve the reported failure and rerun this notebook before starting 05. A repeated reset is destructive to any new results you have since generated. After migration, add YAML experiments and run normal processing; do not reset between batches.

After a successful reset: run the new 05. Update 06/07 before using the new incremental/inference workflow downstream.


In [0]:
# First run in preview mode. Change only RESET_MODE to "execute" when ready
# to discard the old outputs immediately before the first new 05 batch.
RESET_MODE = "preview"  # preview or execute

PIPELINE_SCHEMA = "development_042_silver_sandbox.demand_forecast"
# Explicit allowlist: derived outputs and status records only, newest stage first.
OUTPUT_SUFFIXES = [
    "forecast_selected_configurations", "forecast_error_metrics",
    "forecast_train_test_split", "forecast_evaluation_cohorts", "forecast_evaluation_runs",
    "forecast_neighbors_kept", "demand_forecast_results", "performance_forecast_results", "forecast_runs",
    "like_games_ai_attr", "similarity_run_attempts", "similarity_runs",
]
MLFLOW_PATHS = [
    "/Shared/AI_Feature_Extraction_Evaluation",
    "/Shared/AI_Feature_Extraction_Forecasts",
    "/Shared/AI_Feature_Extraction_Similarity",
]
PROTECTED_SOURCE_SUFFIXES = {
    "game_attr_ai_silver", "sales_silver_comp", "lease_units_bronz", "eilers_group_historical_time_series",
}

if RESET_MODE not in {"preview", "execute"}:
    raise ValueError("RESET_MODE must be preview or execute.")
if set(OUTPUT_SUFFIXES) & PROTECTED_SOURCE_SUFFIXES:
    raise ValueError("The reset allowlist contains a source table; no changes made.")

import pandas as pd
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType


def reset_quote(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))


reset_client = MlflowClient()
reset_tables, reset_runs, reset_inventory_errors = [], [], []
for suffix in OUTPUT_SUFFIXES:
    table_name = PIPELINE_SCHEMA + "." + suffix
    try:
        exists = spark.catalog.tableExists(table_name)
        count = spark.table(table_name).count() if exists else 0
        reset_tables.append({"table": table_name, "exists": exists, "rows": count})
    except Exception as error:
        reset_inventory_errors.append(f"{table_name}: {type(error).__name__}: {error}")

for experiment_path in MLFLOW_PATHS:
    try:
        experiment = reset_client.get_experiment_by_name(experiment_path)
        if experiment is None:
            continue
        page_token = None
        while True:
            page = reset_client.search_runs(
                [experiment.experiment_id], run_view_type=ViewType.ACTIVE_ONLY,
                max_results=1000, page_token=page_token,
            )
            reset_runs.extend({"experiment_path": experiment_path, "run_id": run.info.run_id,
                               "run_status": run.info.status} for run in page)
            page_token = getattr(page, "token", None)
            if not page_token:
                break
    except Exception as error:
        reset_inventory_errors.append(f"{experiment_path}: {type(error).__name__}: {error}")

display(pd.DataFrame(reset_tables))
display(pd.DataFrame(reset_runs, columns=["experiment_path", "run_id", "run_status"]))
if reset_inventory_errors:
    raise RuntimeError("Reset inventory was incomplete. No changes were made.\n" + "\n".join(reset_inventory_errors))
if any(run["run_status"] == "RUNNING" for run in reset_runs):
    print("RUNNING MLflow runs are listed. Check whether they are active work or abandoned runs before executing a reset.")

if RESET_MODE == "preview":
    print("PREVIEW ONLY: no tables or MLflow runs changed.")
    print("Run the execute mode once, immediately before the first new 05 batch, with other pipeline notebooks stopped.")
else:
    reset_results, reset_errors = [], []
    # Delete tracking first. If access fails, retain table history for recovery.
    for item in reset_runs:
        try:
            reset_client.delete_run(item["run_id"])
            reset_results.append({"kind": "mlflow_run", "name": item["run_id"], "status": "deleted"})
        except Exception as error:
            reset_errors.append(f"MLflow {item['run_id']}: {type(error).__name__}: {error}")
    if reset_errors:
        display(pd.DataFrame(reset_results))
        raise RuntimeError("MLflow reset was partial; Delta output tables were retained. Resolve and rerun before starting 05.\n" + "\n".join(reset_errors))
    # Drop/recreate on the next batch, rather than DELETE rows: this also starts
    # fresh schemas and transaction IDs for the deterministic new experiment IDs.
    for item in reset_tables:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {reset_quote(item['table'])}")
            reset_results.append({"kind": "delta_table", "name": item["table"], "status": "removed"})
        except Exception as error:
            reset_errors.append(f"{item['table']}: {type(error).__name__}: {error}")
    display(pd.DataFrame(reset_results))
    if reset_errors:
        raise RuntimeError("Reset was partial. Resolve and rerun before starting 05.\n" + "\n".join(reset_errors))
    remaining_tables = [item["table"] for item in reset_tables if spark.catalog.tableExists(item["table"])]
    if remaining_tables:
        raise RuntimeError(f"Tables remain after reset: {remaining_tables}")
    print("Pipeline outputs and active MLflow run histories were reset. The three MLflow containers and all four source tables were retained.")
    print("Set RESET_MODE back to preview now. Run the new notebook 05 next; it will create fresh output/status tables.")
